# 06 — 意图识别 + 查询改写 交互测试

测试 `backend/intent_recognition/` 模块的三个核心组件：
1. **术语归一化**（Layer 1）— 规则化同义词/缩写映射
2. **意图分类** — 区分设备故障查询与日常对话
3. **查询改写**（Layer 2+3）— LLM改写+拆分，兜底规则拆分

In [24]:
import sys
from pathlib import Path

_project_root = str(Path.cwd().parent)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from backend.intent_recognition import (
    IntentType, IntentResult, RewriteResult, TermMapping,
    classify_intent, rewrite_query, rewrite_with_context,
    QueryTermMappingService, get_term_mapping_service,
)
from backend.intent_recognition.main import process_query

print("✓ 模块导入成功")
print(f"  IntentType: {[e.value for e in IntentType]}")
print(f"  process_query: {process_query}")

✓ 模块导入成功
  IntentType: ['equipment_qa', 'casual_chat']
  process_query: <function process_query at 0x0000022F1D247910>


---
## 1. 术语归一化（Layer 1）

规则化同义词/缩写映射，不依赖 LLM。输入口语化/简写术语，输出规范术语。

In [25]:
# 先看一组内置映射规则
service = get_term_mapping_service()
print("内置的默认映射规则（前 10 条）：")
from backend.intent_recognition.term_mapping import DEFAULT_MAPPINGS
for m in DEFAULT_MAPPINGS[:10]:
    print(f"  {m.source_term:10s} -> {m.target_term}")
print(f"  ... 共 {len(DEFAULT_MAPPINGS)} 条规则")

内置的默认映射规则（前 10 条）：
  电机         -> 电动机
  泵          -> 水泵
  风机         -> 通风机
  空压机        -> 空气压缩机
  压缩机        -> 压缩机
  异响         -> 异常噪音
  漏油         -> 泄漏润滑油
  温升         -> 温度升高
  过热         -> 温度过高
  振动大        -> 异常振动
  ... 共 21 条规则


In [26]:
# 在此输入你想测试的 query，查看归一化结果
query = input("请输入 query: ").strip()

normalized = service.normalize(query)
print(f"\n  original  : {query}")
print(f"  normalized: {normalized}")
if normalized != query:
    print(f"  \u2192 命中规则，已归一化")
else:
    print(f"  \u2192 无匹配规则，保持不变")


  original  : 你好
  normalized: 你好
  → 无匹配规则，保持不变


---
## 2. 意图分类

判断用户输入是 `equipment_qa`（设备故障）还是 `casual_chat`（日常对话）。

In [27]:
query = input("请输入 query: ").strip()

result = classify_intent(query)
print(f"\n  intent    : {result.intent.value}")
print(f"  confidence: {result.confidence:.2%}")
print(f"  explanation: {result.explanation}")
print(f"  needs_rewrite: {result.needs_rewrite}")


  intent    : casual_chat
  confidence: 95.00%
  explanation: 用户询问AI功能，属于日常对话，与设备故障无关
  needs_rewrite: False


---
## 3. 查询改写（Layer 2+3）

三层递进流程：
- Layer 1: 规则归一化（自动执行）
- Layer 2: LLM 改写+拆分（指代消解、短查询扩展、多问句拆分）
- Layer 3: LLM失败 → 规则拆分兜底

输入 query 后，查看完整的改写链路结果。

In [28]:
query = input("请输入 query: ").strip()

result = rewrite_query(query)

print(f"\n  {'='*50}")
print(f"  original      : {result.original_query}")
print(f"  rewritten     : {result.rewritten_query}")
print(f"  reason        : {result.rewrite_reason}")
print(f"  sub_questions : {result.sub_questions}")
print(f"  should_split  : {result.should_split}")
if result.expansion_terms:
    print(f"  expansion     : {', '.join(result.expansion_terms)}")
print(f"\n  {'='*50}")


  original      : 你可以做什么
  rewritten     : 设备故障知识库功能 查询能力
  reason        : 用户询问助手能力，需将抽象问题具体化为知识库功能检索
  sub_questions : ['设备故障知识库功能 查询能力']
  should_split  : False
  expansion     : 故障知识库, 查询助手



---
## 4. 带上下文的改写

在多轮对话场景中测试指代消解。先填写对话历史（可选），再输入当前问题。

In [12]:
history_text = input(
    "对话历史（每行一条，格式：用户:xxx / 助手:xxx，留空跳过）:\n"
).strip()

history = []
if history_text:
    for line in history_text.split("\n"):
        line = line.strip()
        if line.startswith("用户:"):
            history.append({"role": "user", "content": line[3:].strip()})
        elif line.startswith("助手:"):
            history.append({"role": "assistant", "content": line[3:].strip()})
        else:
            # 没有前缀时默认为用户消息
            history.append({"role": "user", "content": line})

if history:
    print(f"\n已设定 {len(history)} 条历史消息:")
    for msg in history:
        print(f"  [{msg['role']}] {msg['content'][:60]}")
else:
    print("\n无历史消息，相当于普通改写")


已设定 1 条历史消息:
  [user] 你好啊


In [15]:
query = input("请输入当前问题: ").strip()

result = rewrite_with_context(query, history=history if history else None)

print(f"\n  {'='*50}")
print(f"  original      : {result.original_query}")
print(f"  rewritten     : {result.rewritten_query}")
print(f"  reason        : {result.rewrite_reason}")
print(f"  sub_questions : {result.sub_questions}")
print(f"  should_split  : {result.should_split}")
if result.expansion_terms:
    print(f"  expansion     : {', '.join(result.expansion_terms)}")
print(f"\n  {'='*50}")


  original      : 怎么解决
  rewritten     : 故障如何解决 处理方法
  reason        : 用户查询过于模糊，无指定设备或故障，补全为通用故障解决方法查询
  sub_questions : ['故障如何解决 处理方法']
  should_split  : False
  expansion     : 故障处理, 解决措施, 维修方法



---
## 5. process_query 管道 — 交互测试

调用 `main.process_query()`，一次展示「归一化 → 意图分类 → 改写」的完整管道输出结果。

In [29]:
query = input("请输入 query: ").strip()

result = process_query(query, verbose=True)

print(f"\n  {'='*50}")
print(f"  原始输入     : {result['original_query']}")
print(f"  归一化后     : {result['normalized_query']}")
print(f"  意图         : {result['intent']} (置信度: {result['confidence']:.2%})")
print(f"  需改写       : {result['needs_rewrite']}")
print(f"  改写结果     : {result['rewritten_query']}")
if result['expansion_terms']:
    print(f"  扩展词       : {', '.join(result['expansion_terms'])}")
if result['should_split']:
    print(f"  拆分后子问题 :")
    for i, sq in enumerate(result['sub_questions'], 1):
        print(f"    {i}. {sq}")
print(f"  总耗时       : {result['latency_ms']:.0f}ms")
print(f"  {'='*50}")

  [1] 输入: 你好
  [2] 术语归一化: 无变化
  [3] 意图分类: casual_chat (置信度=1.00)
  [4] 闲聊意图，跳过改写

  原始输入     : 你好
  归一化后     : 你好
  意图         : casual_chat (置信度: 100.00%)
  需改写       : False
  改写结果     : 你好
  总耗时       : 1618ms


---
## 6. 规则归一化批量测试

一键验证术语映射的覆盖范围和效果。

In [22]:
print("术语归一化覆盖验证 (Layer 1):")
test_cases = [
    "电机温升过高",
    "泵漏油",
    "轴温多少算正常",
    "空压机异响",
    "水压多少正常",
    "振动大怎么办",
    "油温过高怎么回事",
    "这个故障怎么处理",
]

for t in test_cases:
    n = service.normalize(t)
    changed = n != t
    print(f"  {'->' if changed else '  '}  {t:20s}  {n}")

print(f"\n\u2713 测试完成")

术语归一化覆盖验证 (Layer 1):
  ->  电机温升过高                电动机温度升高过高
  ->  泵漏油                   水泵泄漏润滑油
  ->  轴温多少算正常               轴承温度正常工作范围
  ->  空压机异响                 空气压缩机异常噪音
  ->  水压多少正常                冷却水压力多少正常
  ->  振动大怎么办                异常振动处理方法
  ->  油温过高怎么回事              润滑油温度过高原因分析
  ->  这个故障怎么处理              这个故障处理方法

✓ 测试完成


---
## 7. 边界情况快速验证

跑一组典型输入快速确认模块稳定。

In [ ]:
print("意图分类快速验证 (equipment_qa 预期):")
for q in ["离心泵密封泄漏原因", "主轴温度高", "振动值允许范围", "电机异响处理"]:
    r = classify_intent(q)
    ok = r.intent == IntentType.EQUIPMENT_QA
    print(f"  {'OK' if ok else 'FAIL'} [{r.intent.value}] conf={r.confidence:.2f} | {q}")

print("\n意图分类快速验证 (casual_chat 预期):")
for q in ["你好", "今天天气怎么样", "你是谁", "谢谢"]:
    r = classify_intent(q)
    ok = r.intent == IntentType.CASUAL_CHAT
    print(f"  {'OK' if ok else 'FAIL'} [{r.intent.value}] conf={r.confidence:.2f} | {q}")

print("\n查询改写 — 指代消解:")
for q in ["它为什么震动这么大？", "这个故障怎么处理？", "那个轴承还能用吗？"]:
    r = rewrite_query(q)
    print(f"  '{q}' -> {r.rewritten_query}")
    print(f"           sub_qs: {r.sub_questions}")

print("\n查询改写 — 短查询扩展:")
for q in ["电机异响", "振动超标", "漏油", "温度高"]:
    r = rewrite_query(q)
    print(f"  '{q}' -> {r.rewritten_query}")

print("\n✓ 快速验证完成")

print("\n\n=== 8. process_query 管道批量验证 ===")

print("\n--- 设备查询 ---")
test_qs = [
    "离心压缩机振动超标怎么办？",
    "电机异响",
    "那个轴承还能用吗？",
]
for q in test_qs:
    r = process_query(q, verbose=False)
    print(f"  [{r['intent']}] {q}")
    print(f"    rewritten: {r['rewritten_query']}")
    print(f"    terms: {r['expansion_terms'][:3]}")

print("\n--- 闲聊 ---")
for q in ["你好", "你是谁", "今天天气怎么样"]:
    r = process_query(q, verbose=False)
    print(f"  [{r['intent']}] {q}")
    assert r['rewritten_query'] == q

print("\n--- 带上下文的改写 ---")
history = [
    {"role": "user", "content": "离心压缩机振动超标怎么办？"},
    {"role": "assistant", "content": "建议先检查轴承磨损情况。"},
]
r = process_query("那个标准值是多少？", history=history, verbose=False)
print(f"  带历史: 那个标准值是多少？")
print(f"    intent: {r['intent']}")
print(f"    rewritten: {r['rewritten_query']}")
print(f"    terms: {r['expansion_terms'][:3]}")

print("\n--- 空输入与边界 ---")
for q in ["", "  "]:
    r = process_query(q, verbose=False)
    print(f"  [{r['intent']}] {repr(q)[:10]} -> rewritten: {r['rewritten_query']}")

print("\n✓ process_query 管道批量验证完成")

---

**说明**: 第 1-5 节是交互式测试（运行后会等待你输入），第 6-7 节是批量快速验证。